# Week 3: Clean the data and answer questions

## 1. Load the data

In [1]:
import pandas as pd
from pathlib import Path

local_path = Path("..") / "data" / "raw" / "digital_payment_transactions_dirty.xlsx"
url = "https://raw.githubusercontent.com/yuvraj56j-del/Digital-wallet-payment-and-consumer-choice-project/main/data/raw/digital_payment_transactions_dirty.xlsx"

df = pd.read_excel(local_path if local_path.exists() else url)
print("Loaded from:", local_path if local_path.exists() else "GitHub URL")
df_clean = df.copy()

def summarize_missingness(data):
    """Return null count and percentage for each column, largest first."""
    out = pd.DataFrame({
        "null_count": data.isna().sum(),
        "null_percent": (data.isna().mean() * 100).round(2),
    })
    return out.sort_values("null_count", ascending=False)

def summarize_categories(data, column):
    """Return every distinct label in a column with its count (shows spelling variants)."""
    return data[column].value_counts(dropna=False).rename_axis(column).reset_index(name="count")


Loaded from: ..\data\raw\digital_payment_transactions_dirty.xlsx


## 3. Missing values
About 1,330 cells are blank (Week 2 audit). We decide a treatment for each column and give the reason. We **do not drop rows** with blanks, because that would throw away the other good values in those rows.

In [2]:
# Blanks before any treatment
missing_before = summarize_missingness(df_clean)
print("Blank cells before:", int(missing_before["null_count"].sum()))
print("Rows with at least one blank before:", int(df_clean.isna().any(axis=1).sum()))
missing_before

Blank cells before: 1330
Rows with at least one blank before: 1300


,null_count,null_percent
customer_segment,270,1.50
payment_channel,241,1.34
region,205,1.14
merchant_category,169,0.94
day_of_week,144,0.80
amount,117,0.65
transaction_status,108,0.60
timestamp,62,0.34
currency,14,0.08
transaction_id,0,0.00


**Treatment for each column**

| Column | Blanks | Treatment | Reason |
|---|---|---|---|
| merchant_category, payment_channel, customer_segment, region, transaction_status | 169, 241, 270, 205, 108 | Fill with "Unknown" | Each is under 1.5% blank. Filling keeps the row and its other values, and "Unknown" is honest because we do not guess. |
| currency | 14 | Leave blank (section 5) | We do not assume a blank currency is KES. |
| day_of_week | 144 | Rebuild from the timestamp (section 7) | The timestamp is a more reliable source. |
| amount, timestamp | 117, 62 | Leave blank (sections 6 and 7) | We cannot recover a missing amount or date, so we keep the row and exclude it where that value is needed. |

Other options we rejected: **dropping** the rows (loses about 1,300 rows and good data), and **filling with the most common value** (invents data and would inflate one category).

In [3]:
# Fill blanks with "Unknown" in the five categorical columns
fill_unknown_cols = ["merchant_category", "payment_channel", "customer_segment", "region", "transaction_status"]
df_clean[fill_unknown_cols] = df_clean[fill_unknown_cols].fillna("Unknown")
missing_before = summarize_missingness(df)
missing_after = summarize_missingness(df_clean)
comparison = pd.DataFrame({
    "blanks_before": missing_before["null_count"],
    "blanks_after": missing_after["null_count"],
}).sort_values("blanks_before", ascending=False)
print("Rows still in df_clean:", len(df_clean))
comparison



Rows still in df_clean: 18000


,blanks_before,blanks_after
customer_segment,270,0
payment_channel,241,0
region,205,0
merchant_category,169,0
day_of_week,144,144
amount,117,117
transaction_status,108,0
timestamp,62,62
currency,14,14
transaction_id,0,0


**Reading the table:** the five categorical columns now have 0 blanks. The remaining blanks (`amount`, `timestamp`, `currency`, `day_of_week`) are handled in sections 5, 6 and 7.

## 4. Duplicates
We check two things: rows that are **exact copies**, and **repeated transaction IDs**.

In [4]:
print("Fully duplicated rows:", int(df_clean.duplicated().sum()))
print("Repeated transaction_id values (extra copies):", int(df_clean["transaction_id"].duplicated().sum()))

# Flag every row whose ID appears more than once (keep=False marks all copies)
df_clean["id_repeated"] = df_clean["transaction_id"].duplicated(keep=False)
print("Rows flagged id_repeated:", int(df_clean["id_repeated"].sum()))
print("Rows still in df_clean:", len(df_clean))

Fully duplicated rows: 0
Repeated transaction_id values (extra copies): 35
Rows flagged id_repeated: 70
Rows still in df_clean: 18000


In [5]:
# Look at a few repeated IDs side by side
df_clean["id_repeated"] = df_clean["transaction_id"].duplicated(keep=False)

df_clean.loc[df_clean["id_repeated"]].sort_values("transaction_id")[
    ["transaction_id", "timestamp", "merchant_category", "amount", "transaction_status"]
].head(6)

,transaction_id,timestamp,merchant_category,amount,transaction_status
8077,TXN20250001709,2025-10-02 02:11:50,Shopping,2073.89,Successful
11469,TXN20250001709,2025-04-04 10:41:16,Shopping,21970.14,Successful
5257,TXN20250003037,2025-12-15 17:42:41,Government Services,914.34,Successful
5312,TXN20250003037,2025-09-07 12:09:53,Groceries,616.19,Successful
1156,TXN20250003322,2025-03-08 01:06:07,Groceries,235.38,Reversed
8240,TXN20250003322,2025-02-12 15:04:02,Shopping,971.49,Failed


**Decision:** there are no exact duplicate rows, so nothing is removed. The repeated IDs have **different** times, amounts and categories, so they are ID clashes, not copies of the same payment. Deleting one of each pair could remove a real transaction, so we **keep both** and flag them with `id_repeated`.

## 5. Standardise category text and currency
The same category is typed many different ways (capital letters, extra spaces, typos such as `Groceris` or `successfull`). If we leave them, every count and total is split across several labels. We fix this with a **mapping dictionary** built from the labels we saw in Week 2.

Rule used for every column: remove spaces at the ends, make the text lower case, then look it up in the mapping to get the correct label.

In [6]:
def build_mapping(correct_labels, typos=None):
    """Map every lower-case, space-stripped version of a label to its correct label.
    'typos' adds the misspellings (written in lower case) that also map to a correct label."""
    mapping = {label.lower(): label for label in correct_labels}
    mapping["unknown"] = "Unknown"          # blanks filled with "Unknown" earlier stay as they are
    if typos:
        mapping.update(typos)
    return mapping

def standardise(series, mapping):
    """Strip spaces, lower-case, then replace each value by its correct label (blanks stay blank)."""
    return series.str.strip().str.lower().map(mapping)

In [7]:
# The correct labels for each column (from the data dictionary) and the typos seen in the Week 2 audit
rules = {
    "merchant_category": build_mapping(
        ["Groceries", "Food & Dining", "Transport", "Shopping", "Utilities", "Entertainment", "Fuel",
         "Healthcare", "Education", "Cash Transfer", "Subscriptions", "Government Services", "Travel"],
        {"food and dining": "Food & Dining", "food & dinning": "Food & Dining", "groceris": "Groceries",
         "grocery": "Groceries", "transprt": "Transport", "shoping": "Shopping",
         "cash tranfser": "Cash Transfer", "subcription": "Subscriptions"}),
    "payment_channel": build_mapping(
        ["Mobile Wallet", "QR Payment", "USSD", "In-App Payment", "NFC / Contactless", "Web Checkout"],
        {"mobile walet": "Mobile Wallet", "in app payment": "In-App Payment", "in-app paymnt": "In-App Payment",
         "qr paymnt": "QR Payment", "nfc/contactless": "NFC / Contactless", "nfc / contactles": "NFC / Contactless",
         "web chekout": "Web Checkout"}),
    "customer_segment": build_mapping(
        ["Salaried Employee", "Student", "Self-Employed", "Small Business Owner", "Gig Worker", "Retired", "Other"],
        {"salaried employe": "Salaried Employee", "self-employd": "Self-Employed", "self employed": "Self-Employed",
         "small buisness owner": "Small Business Owner", "gig wroker": "Gig Worker", "studnt": "Student"}),
    "region": build_mapping(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Kiambu", "Eldoret", "Machakos", "Nyeri", "Kakamega", "Other"],
        {"nairob": "Nairobi", "nakkuru": "Nakuru", "mombassa": "Mombasa", "kisum": "Kisumu", "eldorret": "Eldoret"}),
    "transaction_status": build_mapping(
        ["Successful", "Failed", "Pending", "Reversed"],
        {"successfull": "Successful", "succesful": "Successful", "pendng": "Pending", "faild": "Failed",
         "reversd": "Reversed"}),
    "day_of_week": build_mapping(
        ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
        {"mon": "Monday", "moday": "Monday", "tues": "Tuesday", "tusday": "Tuesday", "weds": "Wednesday",
         "wensday": "Wednesday", "thur": "Thursday", "thrusday": "Thursday", "fri": "Friday", "fryday": "Friday",
         "sat": "Saturday", "saterday": "Saturday", "sun": "Sunday", "snday": "Sunday"}),
}
print("Mapping rules created for:", list(rules))

Mapping rules created for: ['merchant_category', 'payment_channel', 'customer_segment', 'region', 'transaction_status', 'day_of_week']


In [8]:
# Apply the rules to the working copy and keep a record of what changed
results = []
for col, mapping in rules.items():
    before = df_clean[col]
    after = standardise(before, mapping)

    unmapped = before[before.notna() & after.isna()].unique()      # labels we forgot to map
    cells_changed = int((before.notna() & (before != after)).sum())

    df_clean[col] = after
    results.append({"column": col,
                    "labels_before": before.nunique(),
                    "labels_after": df_clean[col].nunique(),
                    "cells_corrected": cells_changed,
                    "unmapped_labels": list(unmapped)})

pd.DataFrame(results)

,column,labels_before,labels_after,cells_corrected,unmapped_labels
0,merchant_category,33,14,205,[]
1,payment_channel,25,7,323,[]
2,customer_segment,22,8,286,[]
3,region,26,11,229,[]
4,transaction_status,14,5,354,[]
5,day_of_week,28,7,310,[]


**How to read the table:** `labels_before` is the number of different labels in the column before the fix, `labels_after` is the number after. `cells_corrected` is how many cells changed. The word `Unknown` (added by Member 2 for blank values) counts as one label, so a column with 13 correct labels shows 14. `unmapped_labels` must be an empty list `[]` for every column: if a label appears there, add it to that column's typo list and run again.

In [9]:
# Proof: only the intended labels are left in each column
for col in rules:
    print(col, "->", df_clean[col].nunique(), "labels")
    print(df_clean[col].value_counts(dropna=False).to_string())
    print()

merchant_category -> 14 labels
merchant_category
Groceries              3130
Food & Dining          2794
Transport              2287
Shopping               2179
Utilities              1410
Entertainment          1092
Fuel                   1037
Healthcare              891
Education               721
Cash Transfer           719
Subscriptions           606
Government Services     506
Travel                  459
Unknown                 169

payment_channel -> 7 labels
payment_channel
Mobile Wallet        6095
QR Payment           2969
USSD                 2302
In-App Payment       2263
NFC / Contactless    2108
Web Checkout         2022
Unknown               241

customer_segment -> 8 labels
customer_segment
Salaried Employee       5520
Student                 2827
Self-Employed           2824
Small Business Owner    2243
Gig Worker              2169
Other                   1213
Retired                  934
Unknown                  270

region -> 11 labels
region
Nairobi     5638
Mombasa 

## Currency
`currency` has `KES` written several ways (`kes`, `KES ` with a space, `Ksh`) and 6 rows in `USD`. We standardise the KES variants. We **do not convert USD to KES** (we have no exchange rate in the data), so USD rows are flagged and kept out of KES totals. The 14 blank currencies stay blank: we do not assume they are KES.

In [10]:
currency_map = {"kes": "KES", "ksh": "KES", "usd": "USD"}

print("Currency labels before:")
print(df_clean["currency"].value_counts(dropna=False).to_string())

df_clean["currency"] = standardise(df_clean["currency"], currency_map)
df_clean["non_kes"] = df_clean["currency"].eq("USD")      # flag, do not delete

print()
print("Currency labels after:")
print(df_clean["currency"].value_counts(dropna=False).to_string())
print()
print("Rows flagged non_kes (USD):", int(df_clean["non_kes"].sum()))
print("Rows still in df_clean:", len(df_clean))

Currency labels before:
currency
KES     17960
NaN        14
kes        10
Ksh         7
USD         6
KES         3

Currency labels after:
currency
KES    17980
NaN       14
USD        6

Rows flagged non_kes (USD): 6
Rows still in df_clean: 18000


## 6. Clean the amounts and flag outliers
We fix the one obvious typo, convert `amount` to numbers, and **flag** bad values in new columns. Nothing is deleted: all rows stay.

In [11]:
# What the raw column contains (before any change)
typo_count = int((df["amount"] == "1,2OO").sum())
unknown_count = int((df["amount"] == "unknown").sum())
print("Entries '1,2OO' (comma and letter O):", typo_count)
print("Entries 'unknown':", unknown_count)

Entries '1,2OO' (comma and letter O): 8
Entries 'unknown': 6


In [12]:
# 1. "1,2OO" is an obvious typo for 1200, so we correct it. "unknown" has no value, so it becomes blank.
amount_fixed = df_clean["amount"].replace({"1,2OO": "1200"})
df_clean["amount"] = pd.to_numeric(amount_fixed, errors="coerce")

print("Amount data type after cleaning:", df_clean["amount"].dtype)
print("Blank amounts before:", int(df["amount"].isna().sum()), "| after:", int(df_clean["amount"].isna().sum()))

Amount data type after cleaning: float64
Blank amounts before: 117 | after: 123


**Why we chose these fixes:** `1,2OO` is clearly 1,200 with a comma and the letter O typed instead of zero, so we correct it. `unknown` gives us nothing to recover, so it becomes blank. Both choices are in `docs/cleaning_log.md`.

In [13]:
print("Largest 12 amounts:")
print(df_clean["amount"].nlargest(12).tolist())
print()
print("99.9th percentile:", round(df_clean["amount"].quantile(0.999), 2))

Largest 12 amounts:
[999999999.0, 999999999.0, 999999999.0, 999999999.0, 999999999.0, 180000.0, 162956.02, 153522.57, 66241.32, 64132.36, 62461.67, 61263.24]

99.9th percentile: 47909.38


**Reading the output:** the largest realistic-looking amounts stop at about KES 66,000. After a clear gap come 153,522.57, 162,956.02, 180,000 and five values of 999,999,999. So we set the outlier limit at 100,000 KES, which separates the extreme values from the rest. Write your own reason in `docs/cleaning_log.md`.

In [14]:
# 2. Flagging (not deleting) invalid and extreme amounts
OUTLIER_LIMIT = 100000      # KES: our rule, explained in the cleaning log

df_clean["amount_invalid"] = df_clean["amount"] <= 0              # negative or zero
df_clean["is_outlier"] = df_clean["amount"] > OUTLIER_LIMIT       # extreme high values

print("Negative amounts:", int((df_clean["amount"] < 0).sum()))
print("Zero amounts:", int((df_clean["amount"] == 0).sum()))
print("Flagged amount_invalid (negative or zero):", int(df_clean["amount_invalid"].sum()))
print("Flagged is_outlier (above", f"{OUTLIER_LIMIT:,} KES):", int(df_clean["is_outlier"].sum()))
print("Rows still in df_clean:", len(df_clean))

Negative amounts: 15
Zero amounts: 7
Flagged amount_invalid (negative or zero): 22
Flagged is_outlier (above 100,000 KES): 8
Rows still in df_clean: 18000


In [15]:
# Effect of the flags: the same statistics with and without flagged values
valid = df_clean[~df_clean["amount_invalid"] & ~df_clean["is_outlier"]]["amount"]

print("All numeric amounts : median", round(df_clean["amount"].median(), 2), "| mean", round(df_clean["amount"].mean(), 2))
print("Valid amounts only  : median", round(valid.median(), 2), "| mean", round(valid.mean(), 2), "| count", valid.count())

All numeric amounts : median 1008.84 | mean 281680.48
Valid amounts only  : median 1009.35 | mean 1967.23 | count 17847


Compare the two lines above: the mean drops from several hundred thousand shillings to about KES 2,000 once flagged values are excluded, while the median barely changes. This is why we report medians and exclude flagged rows from value analysis.

## 7. Parse the timestamps and create date columns
We convert `timestamp` to a real date and time. Values that cannot be read become blank (`NaT`); we keep those rows for the analysis that does not need a date.

In [16]:
raw_blank_ts = int(df["timestamp"].isna().sum())
raw_invalid_ts = int((pd.to_datetime(df["timestamp"], errors="coerce").isna() & df["timestamp"].notna()).sum())

df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"], errors="coerce")

print("Blank timestamps in raw data:", raw_blank_ts)
print("Invalid timestamps in raw data:", raw_invalid_ts)
print("Blank (NaT) timestamps after cleaning:", int(df_clean["timestamp"].isna().sum()))
print("Earliest:", df_clean["timestamp"].min())
print("Latest:  ", df_clean["timestamp"].max())

Blank timestamps in raw data: 62
Invalid timestamps in raw data: 22
Blank (NaT) timestamps after cleaning: 84
Earliest: 2025-01-01 00:31:22
Latest:   2025-12-31 23:45:05


In [17]:
# Date parts used later for the time-trend questions
df_clean["month"] = df_clean["timestamp"].dt.month.astype("Int64")   # whole numbers; blank where no valid date
df_clean["hour"] = df_clean["timestamp"].dt.hour.astype("Int64")
df_clean["weekday"] = df_clean["timestamp"].dt.day_name()

df_clean[["timestamp", "month", "hour", "weekday"]].head()

,timestamp,month,hour,weekday
0,2025-05-13 12:18:54,5,12,Tuesday
1,2025-01-29 11:29:05,1,11,Wednesday
2,2025-11-28 04:54:27,11,4,Friday
3,2025-12-05 14:13:05,12,14,Friday
4,2025-09-09 08:58:53,9,8,Tuesday


In [18]:
# Rebuild day_of_week from the timestamp where the timestamp is valid.
# First check how often the cleaned day_of_week disagrees with the timestamp.
both = df_clean["weekday"].notna() & df_clean["day_of_week"].notna()
mismatch = int((df_clean.loc[both, "day_of_week"] != df_clean.loc[both, "weekday"]).sum())
print("Rows where day_of_week and the timestamp both exist:", int(both.sum()))
print("Rows where they disagree:", mismatch)

df_clean["day_of_week"] = df_clean["weekday"].fillna(df_clean["day_of_week"])
print("Blank day_of_week after rebuilding:", int(df_clean["day_of_week"].isna().sum()))

Rows where day_of_week and the timestamp both exist: 17772
Rows where they disagree: 0
Blank day_of_week after rebuilding: 0


**What this section does:** the timestamp is more reliable than the typed weekday, so where a valid timestamp exists we take the weekday from it; otherwise we keep the cleaned `day_of_week`. Rows with no usable timestamp stay in the data but are left out of the month, hour and weekday questions.

## 8. Save and validate the processed dataset
We save `df_clean` to `data/processed/` (the raw file is untouched), reload it from disk, and prove it still has at least 15,000 rows and 10 useful columns.

In [19]:
from pathlib import Path

def meets_size(data, min_rows=15000, min_cols=10):
    """Return True if the table has enough rows and columns."""
    rows, cols = data.shape
    return rows >= min_rows and cols >= min_cols

PROCESSED_PATH = Path("..") / "data" / "processed" / "digital_payment_transactions_cleaned.csv"
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean["transaction_id"] = df_clean["transaction_id"].str.strip()
df_clean.to_csv(PROCESSED_PATH, index=False)

saved = pd.read_csv(PROCESSED_PATH)
print("File saved to:", PROCESSED_PATH.resolve())
print("Raw data shape:", df.shape)
print("Processed file shape:", saved.shape)
print("Meets 15,000 x 10 requirement:", meets_size(saved))

File saved to: C:\Users\ADMIN\Documents\Digital-wallet-payment-and-consumer-choice-project\data\processed\digital_payment_transactions_cleaned.csv
Raw data shape: (18000, 10)
Processed file shape: (18000, 17)
Meets 15,000 x 10 requirement: True


In [20]:
df_clean["transaction_id"] = df_clean["transaction_id"].str.strip()
df_clean["id_repeated"] = df_clean["transaction_id"].duplicated(keep=False)
print("Rows flagged id_repeated:", int(df_clean["id_repeated"].sum()))

Rows flagged id_repeated: 70


In [21]:
valid_labels = {
    "merchant_category": ["Groceries", "Food & Dining", "Transport", "Shopping", "Utilities", "Entertainment",
                          "Fuel", "Healthcare", "Education", "Cash Transfer", "Subscriptions",
                          "Government Services", "Travel", "Unknown"],
    "payment_channel": ["Mobile Wallet", "QR Payment", "USSD", "In-App Payment", "NFC / Contactless",
                        "Web Checkout", "Unknown"],
    "customer_segment": ["Salaried Employee", "Student", "Self-Employed", "Small Business Owner",
                         "Gig Worker", "Retired", "Other", "Unknown"],
    "region": ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Kiambu", "Eldoret", "Machakos", "Nyeri",
               "Kakamega", "Other", "Unknown"],
    "transaction_status": ["Successful", "Failed", "Pending", "Reversed", "Unknown"],
    "day_of_week": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
    "currency": ["KES", "USD"],
}

checks = {}
checks["At least 15,000 rows"] = len(df_clean) >= 15000
checks["At least 10 columns"] = df_clean.shape[1] >= 10
for col, labels in valid_labels.items():
    checks[f"{col}: only valid labels"] = bool(df_clean[col].dropna().isin(labels).all())
for col in ["merchant_category", "payment_channel", "customer_segment", "region",
            "transaction_status", "day_of_week", "transaction_id"]:
    checks[f"{col}: no blanks"] = int(df_clean[col].isna().sum()) == 0
for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    text = df_clean[col].dropna().astype(str)
    checks[f"{col}: no extra spaces"] = bool((text == text.str.strip()).all())
checks["amount is numeric"] = pd.api.types.is_numeric_dtype(df_clean["amount"])
checks["no unflagged zero/negative amounts"] = not ((df_clean["amount"] <= 0) & ~df_clean["amount_invalid"]).any()
checks["no unflagged huge amounts"] = not ((df_clean["amount"] > OUTLIER_LIMIT) & ~df_clean["is_outlier"]).any()
checks["timestamp is datetime"] = pd.api.types.is_datetime64_any_dtype(df_clean["timestamp"])
checks["all timestamps in 2025"] = bool(df_clean["timestamp"].dropna().dt.year.eq(2025).all())
both = df_clean["timestamp"].notna() & df_clean["day_of_week"].notna()
checks["day_of_week matches timestamp"] = bool((df_clean.loc[both, "day_of_week"] == df_clean.loc[both, "timestamp"].dt.day_name()).all())
checks["no fully duplicated rows"] = int(df_clean.duplicated().sum()) == 0
checks["repeated IDs are all flagged"] = not (df_clean["transaction_id"].duplicated(keep=False) & ~df_clean["id_repeated"]).any()

report = pd.DataFrame({"check": list(checks), "result": ["PASS" if v else "FAIL" for v in checks.values()]})
print("PASS:", (report["result"] == "PASS").sum(), "| FAIL:", (report["result"] == "FAIL").sum())
print("df_clean shape:", df_clean.shape)
report[report["result"] == "FAIL"]

PASS: 33 | FAIL: 0
df_clean shape: (18000, 17)


,check,result


## 9. Before/after cleaning summary

In [22]:
core = list(df.columns)
cat_cols = ["merchant_category", "payment_channel", "customer_segment", "region", "transaction_status", "day_of_week"]

# Fields corrected = non-blank raw values that are different after cleaning
corrected = {}
for col in cat_cols + ["currency"]:
    corrected[col] = int((df[col].notna() & (df[col].astype("string") != df_clean[col].astype("string"))).sum())
corrected["amount (1,2OO repaired)"] = int((df["amount"].astype("string") == "1,2OO").sum())
corrected["transaction_id (spaces removed)"] = int((df["transaction_id"].astype("string") != df_clean["transaction_id"].astype("string")).sum())

summary = pd.DataFrame({
    "Before cleaning": [
        len(df), df.shape[1], int(df[core].isna().sum().sum()), int(df[core].isna().any(axis=1).sum()),
        int(df.duplicated().sum()), int(df["transaction_id"].astype("string").str.strip().duplicated().sum()),
        0, 0, 0,
    ],
    "After cleaning": [
        len(df_clean), df_clean.shape[1], int(df_clean[core].isna().sum().sum()), int(df_clean[core].isna().any(axis=1).sum()),
        int(df_clean.duplicated().sum()), int(df_clean["transaction_id"].duplicated().sum()),
        int((df_clean[cat_cols] == "Unknown").sum().sum()), int(df_clean["amount_invalid"].sum()), int(df_clean["is_outlier"].sum()),
    ],
}, index=["Rows", "Columns", "Null cells (10 original columns)", "Rows with at least one null",
          "Fully duplicated rows", "Repeated transaction IDs (flagged, kept)",
          "Blanks labelled 'Unknown'", "Invalid amounts flagged", "Outliers flagged"])

display(summary)
print("Rows retained:", len(df_clean), "of", len(df), f"({len(df_clean) / len(df) * 100:.2f}%)")
display(pd.Series(corrected, name="values_corrected").to_frame())

,Before cleaning,After cleaning
Rows,18000,18000
Columns,10,17
Null cells (10 original columns),1330,221
Rows with at least one null,1300,219
Fully duplicated rows,0,0
"Repeated transaction IDs (flagged, kept)",35,35
Blanks labelled 'Unknown',0,993
Invalid amounts flagged,0,22
Outliers flagged,0,8


Rows retained: 18000 of 18000 (100.00%)


,values_corrected
merchant_category,205
payment_channel,323
customer_segment,286
region,229
transaction_status,354
day_of_week,310
currency,20
"amount (1,2OO repaired)",8
transaction_id (spaces removed),45


### Question 1: Which merchant categories account for the largest share of total spending in KES?

In [23]:
# Q1: Rows used for spending analysis: successful payments with a usable KES amount
spending = df_clean[
    (df_clean["transaction_status"] == "Successful")
    & df_clean["amount"].notna()
    & ~df_clean["amount_invalid"]
    & ~df_clean["is_outlier"]
    & (df_clean["currency"] == "KES")
]
print("Rows in spending:", len(spending), "of", len(df_clean))

Rows in spending: 16381 of 18000


In [24]:
q1 = (spending.groupby("merchant_category")["amount"]
      .agg(total_kes="sum", transactions="count")
      .sort_values("total_kes", ascending=False))
q1["share_percent"] = (q1["total_kes"] / q1["total_kes"].sum() * 100).round(1)
q1["total_kes"] = q1["total_kes"].round(0)
q1

,total_kes,transactions,share_percent
merchant_category,,,
Shopping,6970552.0,1988,21.6
Travel,4846905.0,423,15.0
Utilities,3179092.0,1279,9.8
Groceries,3141596.0,2846,9.7
Cash Transfer,2537121.0,647,7.9
Food & Dining,2236592.0,2552,6.9
Education,2189496.0,658,6.8
Fuel,2038488.0,932,6.3
Healthcare,1574584.0,813,4.9


**Interpretation:** Shopping accounts for the largest share of spending: KES 6,970,552.0, or 21.6% of the KES 32.3 million total. Travel is second with 15.0% even though it has only 423 transactions, because its payments are large. Groceries has the most transactions (2,846) but only 9.7% of the value, so frequent categories are not always the biggest spenders. Shopping, Travel, Utilities and Groceries together make up about 56% of spending.

### Question 2: Which customer segments have the highest median transaction value?

In [25]:
q2 = (spending.groupby("customer_segment")["amount"]
      .agg(median_kes="median", mean_kes="mean", transactions="count")
      .sort_values("median_kes", ascending=False)
      .round(2))
q2

,median_kes,mean_kes,transactions
customer_segment,,,
Student,1052.30,2070.94,2593
Small Business Owner,1044.11,1990.60,2040
Gig Worker,1026.57,1893.07,1971
Self-Employed,1011.54,2039.99,2555
Retired,1003.35,1859.62,841
Salaried Employee,989.24,1948.69,5036
Unknown,983.75,1705.74,248
Other,968.39,1927.36,1097


**Interpretation:** Students have the highest median transaction value (KES 1,052.30), followed by Small Business Owners (1,044.11), and the lowest named segment is Other (968.39). The gap between highest and lowest is only about KES 84, so customer segment makes little difference to a typical payment. Salaried Employees are the largest group (5,036 spending transactions) with a median of 989.24. With such small differences, we should not claim any segment spends clearly more.

### Question 3: Which regions record the most transactions, and which have the highest average value?

In [26]:
# Transactions: every row in the region (all statuses). Value: spending rows only.
q3 = pd.DataFrame({
    "transactions": df_clean.groupby("region").size(),
    "avg_value_kes": spending.groupby("region")["amount"].mean(),
    "median_value_kes": spending.groupby("region")["amount"].median(),
}).sort_values("transactions", ascending=False).round(2)
q3

,transactions,avg_value_kes,median_value_kes
region,,,
Nairobi,5638,1996.91,1013.58
Mombasa,2533,1938.11,965.98
Kisumu,1772,1963.81,1006.34
Nakuru,1589,2179.50,1082.20
Kiambu,1448,1924.40,1036.28
Eldoret,1351,1946.61,1048.86
Other,1176,1919.63,985.87
Machakos,1074,1899.00,988.64
Nyeri,684,1645.25,971.54


**Interpretation:** Nairobi records the most transactions by a wide margin (5,638), followed by Mombasa (2,533) and Kisumu (1,772); Kakamega (530) and Nyeri (684) have the fewest. Nakuru has the highest average value (KES 2,179.50) and also the highest median (1,082.20), followed by Kakamega (2,077.58). Nairobi's average (1,996.91) is third among the named regions. The medians for all regions fall between about 966 and 1082, so the differences in averages are partly caused by a few large payments.

### Question 4: How do total monthly transaction value and transaction count change across 2025?

In [27]:
q4 = (spending.groupby("month")["amount"]
      .agg(total_kes="sum", transactions="count")
      .round(0))
q4.index = q4.index.astype(int)
q4.index.name = "month"
print("Spending rows with no valid date (excluded from this table):", int(spending["month"].isna().sum()))
q4

Spending rows with no valid date (excluded from this table): 78


,total_kes,transactions
month,,
1,2507385.0,1339
2,2319377.0,1265
3,2712231.0,1357
4,2700056.0,1335
5,2794108.0,1394
6,2700850.0,1338
7,2637456.0,1379
8,2817615.0,1429
9,2652959.0,1378


**Interpretation:** Monthly spending stays fairly steady through 2025, between KES 2.32 million in February and KES 2.82 million in December (August is close behind at 2.82 million). Transaction counts range from 1,265 in February to 1,429 in August. February is the lowest on both measures, which fits it being the shortest month. We see no strong upward or downward trend. The 78 spending rows without a valid date are not included in this table.

### Question 5: Which Weekday and Which Hour of the Day Are the Busiest?

In [28]:
# Q5. Busiest weekday and hour (count of transactions with a valid timestamp)
timed = df_clean[df_clean["timestamp"].notna()]
print("Transactions with a valid timestamp:", len(timed), "of", len(df_clean))

# Weekday
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = timed["weekday"].value_counts().reindex(day_order).rename("transactions").to_frame()
by_day["share_%"] = (by_day["transactions"] / by_day["transactions"].sum() * 100).round(1)
print("Busiest weekday:", by_day["transactions"].idxmax(), "| quietest:", by_day["transactions"].idxmin())
by_day

Transactions with a valid timestamp: 17916 of 18000
Busiest weekday: Sunday | quietest: Thursday


,transactions,share_%
weekday,,
Monday,2595,14.5
Tuesday,2558,14.3
Wednesday,2597,14.5
Thursday,2464,13.8
Friday,2563,14.3
Saturday,2514,14.0
Sunday,2625,14.7


In [29]:
# Hour
by_hour = timed["hour"].value_counts().sort_index().rename("transactions").to_frame()
by_hour["share_%"] = (by_hour["transactions"] / by_hour["transactions"].sum() * 100).round(1)
print("Busiest hour:", by_hour["transactions"].idxmax(), "| quietest:", by_hour["transactions"].idxmin())
print()
print("Top 5 busiest hours:")
print(by_hour.sort_values("transactions", ascending=False).head(5))
print()
print("Quietest 3 hours:")
print(by_hour.sort_values("transactions").head(3))

Busiest hour: 17 | quietest: 10

Top 5 busiest hours:
      transactions  share_%
hour                       
17             790      4.4
18             786      4.4
0              785      4.4
11             780      4.4
13             779      4.3

Quietest 3 hours:
      transactions  share_%
hour                       
10             676      3.8
23             707      3.9
14             717      4.0


## Interpretation

Based on all transactions with a valid timestamp, **Sunday was the busiest weekday**, recording **2,625 transactions (14.7%)**, while **Thursday was the least busy**, with **2,464 transactions (13.8%)**.

Looking at the time of day, **17:00 (5 PM)** had the highest number of transactions with **790 transactions**, while **10:00 AM** had the lowest with **676 transactions**.

However, the differences between weekdays and hours are relatively small. For example, Sunday recorded only about **6.5% more transactions than Thursday**. Similarly, the five busiest hours were separated by only **11 transactions**, showing very little variation throughout the day.

These results suggest that transaction activity is fairly evenly distributed across the week and across different hours of the day. Therefore, it would be inaccurate to conclude that there is a strong **rush hour** or a significant **weekend effect** based on this data.

It is also important to note that the results depend on the data being analyzed. In a previous analysis that considered only successful transactions, **Monday and Sunday were tied as the busiest weekdays**. Therefore, this finding specifically applies to **all transactions with a valid timestamp**.

### Question 6: Which Payment Channels Have the Highest Failure Rate?

In [30]:
# Q6. Failure rate by payment channel
# Use only rows where both the channel and the status are known
known = df_clean[(df_clean["transaction_status"] != "Unknown") & (df_clean["payment_channel"] != "Unknown")]
print("Rows used:", len(known), "of", len(df_clean))

grouped = known.groupby("payment_channel")["transaction_status"]
fail_by_channel = pd.DataFrame({
    "transactions": grouped.size(),
    "failed": grouped.apply(lambda s: int((s == "Failed").sum())),
})
fail_by_channel["failure_rate_%"] = (fail_by_channel["failed"] / fail_by_channel["transactions"] * 100).round(2)
fail_by_channel = fail_by_channel.sort_values("failure_rate_%", ascending=False)

overall = (known["transaction_status"] == "Failed").mean() * 100
print(f"Overall failure rate: {overall:.2f}%")
fail_by_channel

Rows used: 17651 of 18000
Overall failure rate: 3.45%


,transactions,failed,failure_rate_%
payment_channel,,,
In-App Payment,2248,88,3.91
USSD,2285,81,3.54
Web Checkout,2008,70,3.49
Mobile Wallet,6062,209,3.45
NFC / Contactless,2096,71,3.39
QR Payment,2952,90,3.05


## Interpretation

The analysis shows that **In-App Payment** had the highest failure rate at **3.91%**, followed by **USSD** at **3.54%**. **QR Payment** recorded the lowest failure rate at **3.05%**.

The overall failure rate across all payment channels was **3.45%**. **Mobile Wallet**, which processed the largest number of transactions (**6,062 transactions**), had a failure rate that was very close to this overall average.

Although In-App Payment had the highest failure rate in this dataset, the difference between the highest and lowest rates was less than **1 percentage point** (about **0.9 percentage points**). This indicates that the performance of the payment channels was fairly similar.

In addition, each channel recorded only **70 to 209 failed transactions**, meaning the observed differences are relatively small and could be due to normal variation in the data. Therefore, it would not be appropriate to conclude that one payment channel is significantly less reliable than the others.

Overall, the results suggest that no payment channel stands out as clearly unreliable. Instead, the failure rates are broadly comparable across all channels. If this dataset is synthetic, these findings should be interpreted as reflecting the generated data and not the real-world reliability of the payment methods.

### Question 7: Do Failed or Reversed Transactions Differ in Amount from Successful Ones?


In [31]:
# Q7. Do failed or reversed transactions differ in amount from successful ones?
valid = df_clean[
    df_clean["amount"].notna()
    & ~df_clean["amount_invalid"]
    & ~df_clean["is_outlier"]
    & ~df_clean["non_kes"]
    & (df_clean["transaction_status"] != "Unknown")
]
print("Rows used:", len(valid), "of", len(df_clean))

status_order = ["Successful", "Failed", "Reversed", "Pending"]
amount_by_status = (valid.groupby("transaction_status")["amount"]
                    .agg(transactions="count", mean="mean", median="median", min="min", max="max")
                    .reindex(status_order).round(2))
amount_by_status

Rows used: 17733 of 18000


,transactions,mean,median,min,max
transaction_status,,,,,
Successful,16392,1971.21,1011.50,20.69,66241.32
Failed,611,1929.37,1083.37,51.59,40292.47
Reversed,362,2111.69,975.36,71.92,32508.70
Pending,368,1686.24,920.03,58.24,28478.82


In [32]:
# Compare each status with Successful (percentage difference)
base = amount_by_status.loc["Successful"]
comparison = pd.DataFrame({
    "median_diff_%": ((amount_by_status["median"] - base["median"]) / base["median"] * 100).round(1),
    "mean_diff_%": ((amount_by_status["mean"] - base["mean"]) / base["mean"] * 100).round(1),
}).drop("Successful")
comparison


,median_diff_%,mean_diff_%
transaction_status,,
Failed,7.1,-2.1
Reversed,-3.6,7.1
Pending,-9.0,-14.5


## Interpretation

The results show only small differences in transaction amounts across the three transaction statuses.

Failed transactions had a slightly higher typical amount, with a median of **1,083 KES**, compared to **1,012 KES** for successful transactions. However, the average amount for failed transactions (**1,929 KES**) was slightly lower than that of successful transactions (**1,971 KES**).

Reversed transactions showed the opposite trend. They had a lower median amount (**975 KES**) but a higher average amount (**2,112 KES**) than successful transactions.

Because the median and average tell different stories, and the differences are quite small, there is no clear evidence that failed or reversed transactions involve consistently higher or lower amounts than successful transactions.

Overall, transaction amounts appear to be fairly similar across successful, failed, and reversed transactions.